
# A New Markov Chain Example: Weather + Subscription Churn

This notebook complements `markov_chain_colab.ipynb` (the ninja demo) and
`markov_chains_tutorial.md`. It walks through **two fresh examples**:

1. A 3-state **weather chain** — a regular chain, to numerically confirm the
   worked example from the tutorial.
2. A **customer subscription churn** model — an *absorbing* Markov chain,
   which behaves very differently from a regular chain and introduces the
   "fundamental matrix" technique for absorption probabilities.

Run the setup cell first (it re-defines the same generic helpers used in the
ninja notebook, so this notebook works standalone).


In [ ]:

import numpy as np
import matplotlib.pyplot as plt
import networkx as nx
from IPython.display import clear_output
import time

def plot_markov_diagram(ax, M, node_values, labels=None, cmap="Blues", title=""):
    n = M.shape[0]
    if labels is None:
        labels = [f"S{i}" for i in range(n)]
    G = nx.DiGraph()
    for i in range(n):
        G.add_node(i)
    for i in range(n):
        for j in range(n):
            if M[i, j] > 1e-9:
                G.add_edge(i, j, weight=M[i, j])
    pos = nx.circular_layout(G)
    cmap_obj = plt.get_cmap(cmap)
    vals = np.asarray(node_values, dtype=float)
    vmax = max(vals.max(), 1e-9)
    node_colors = cmap_obj(0.25 + 0.65 * (vals / vmax))
    nx.draw_networkx_nodes(G, pos, ax=ax, node_size=1900, node_color=node_colors,
                            edgecolors="black", linewidths=1.5)
    nx.draw_networkx_labels(
        G, pos, ax=ax,
        labels={i: f"{labels[i]}\n{node_values[i]:.2f}" for i in range(n)},
        font_size=9)
    self_loops = [(u, v) for u, v in G.edges() if u == v]
    normal_edges = [(u, v) for u, v in G.edges() if u != v]
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=normal_edges,
                            connectionstyle="arc3,rad=0.15", arrowsize=15,
                            width=1.4, edge_color="gray")
    nx.draw_networkx_edges(G, pos, ax=ax, edgelist=self_loops,
                            connectionstyle="arc3,rad=0.6", arrowsize=15,
                            width=1.4, edge_color="gray")
    edge_labels = {(u, v): f"{M[u, v]:.2f}" for u, v in G.edges()}
    nx.draw_networkx_edge_labels(G, pos, ax=ax, edge_labels=edge_labels, font_size=8)
    ax.set_title(title)
    ax.axis("off")

def stationary_distribution(P):
    evals, evecs = np.linalg.eig(P.T)
    idx = np.argmin(np.abs(evals - 1))
    vec = np.real(evecs[:, idx])
    return vec / vec.sum()



## Part 1 — Weather chain (regular, converges)

Three states: **Sunny**, **Cloudy**, **Rainy**.


In [ ]:
weather_labels = ["Sunny", "Cloudy", "Rainy"]

P_weather = np.array([
    [0.70, 0.25, 0.05],   # from Sunny
    [0.30, 0.40, 0.30],   # from Cloudy
    [0.15, 0.35, 0.50],   # from Rainy
])

assert np.allclose(P_weather.sum(axis=1), 1), "rows must sum to 1"

pi = stationary_distribution(P_weather)
print("Stationary distribution [Sunny, Cloudy, Rainy]:", np.round(pi, 3))


In [ ]:
# Confirm it two ways: brute-force power vs. eigenvector method
P_100 = np.linalg.matrix_power(P_weather, 100)
print("Every row of P^100 (brute force):\n", np.round(P_100, 3))
print("\nEigenvector method:", np.round(pi, 3))


In [ ]:
# Try 3 very different starting beliefs -- they should all converge to the SAME pi
starts = {
    "Certain it's sunny today":  np.array([1.0, 0.0, 0.0]),
    "Certain it's rainy today":  np.array([0.0, 0.0, 1.0]),
    "No idea (uniform guess)":   np.array([1/3, 1/3, 1/3]),
}

fig, ax = plt.subplots(figsize=(7, 5))
for name, x0 in starts.items():
    trace = [x0]
    x = x0.copy()
    for _ in range(15):
        x = x @ P_weather
        trace.append(x)
    trace = np.array(trace)
    ax.plot(trace[:, 0], label=f"{name} (P[Sunny])", marker=".")

ax.axhline(pi[0], color="gray", linestyle="--", label="stationary P[Sunny]")
ax.set_xlabel("days")
ax.set_ylabel("P(Sunny)")
ax.set_title("All starting beliefs converge to the same stationary distribution")
ax.legend()
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))
plot_markov_diagram(ax, P_weather, pi, labels=weather_labels, title="Weather chain (node = stationary prob.)")
plt.tight_layout()
plt.show()



## Part 2 — Subscription churn (an *absorbing* chain)

Now a genuinely different flavor of Markov chain. States:
**Trial → Active → At-Risk → Churned**. Once a customer reaches
`Churned`, they never come back — that's an **absorbing state**
(`P(Churned → Churned) = 1`).

Absorbing chains don't converge to an interesting stationary distribution the
way regular chains do — everyone eventually ends up in an absorbing state
(here, just `Churned`, since there's no "loyal forever" state in this toy
model). The interesting questions instead are:

- **How many months, on average, before a new trial user churns?**
- **What's the probability of ever reaching "Active", starting from "Trial"?**


In [ ]:
churn_labels = ["Trial", "Active", "At-Risk", "Churned"]

P_churn = np.array([
    [0.10, 0.60, 0.20, 0.10],   # Trial
    [0.00, 0.80, 0.15, 0.05],   # Active
    [0.00, 0.30, 0.40, 0.30],   # At-Risk
    [0.00, 0.00, 0.00, 1.00],   # Churned (absorbing)
])

assert np.allclose(P_churn.sum(axis=1), 1)

fig, ax = plt.subplots(figsize=(6, 6))
plot_markov_diagram(ax, P_churn, P_churn.diagonal(), labels=churn_labels,
                     cmap="Reds", title="Subscription churn chain")
plt.tight_layout()
plt.show()



### The fundamental matrix

Split `P` into blocks based on transient states `Q` (Trial, Active, At-Risk)
and the absorbing state:

```
P = [ Q   R ]
    [ 0   1 ]
```

The **fundamental matrix** is `N = (I − Q)⁻¹`. Two things fall out of it for free:

- **Expected number of steps until absorption**, starting from state `i`:
  row sums of `N`.
- **Absorption probabilities** into each absorbing state: `N @ R`
  (trivial here since there's only one absorbing state — it's just `1`
  everywhere, but the math generalizes to multiple absorbing states, e.g.
  "Churned" vs. "Upgraded to annual plan").


In [ ]:
Q = P_churn[:3, :3]                 # transient-to-transient block
I = np.eye(3)
N = np.linalg.inv(I - Q)             # fundamental matrix

expected_steps = N.sum(axis=1)       # expected months until churn, per starting state
print("Fundamental matrix N:\n", np.round(N, 2))
print("\nExpected months until churn, starting from:")
for label, val in zip(churn_labels[:3], expected_steps):
    print(f"  {label:10s}: {val:.2f} months")


In [ ]:
# Sanity-check with brute-force simulation (Monte Carlo)
rng = np.random.default_rng(0)

def simulate_until_churn(start_state, P, rng, max_steps=1000):
    state = start_state
    steps = 0
    while state != 3 and steps < max_steps:  # 3 = Churned
        state = rng.choice(4, p=P[state])
        steps += 1
    return steps

n_trials = 20000
sim_avg = np.mean([simulate_until_churn(0, P_churn, rng) for _ in range(n_trials)])
print(f"Monte Carlo estimate (starting from Trial, {n_trials} runs): {sim_avg:.2f} months")
print(f"Exact (fundamental matrix) answer: {expected_steps[0]:.2f} months")



The simulation and the exact fundamental-matrix answer should match closely —
that's a nice way to sanity-check both the math and the code any time you
build a new chain.

## Takeaways

- **Regular chains** (weather) converge to one long-run distribution no
  matter where you start — great for "what's the long-run behavior" questions.
- **Absorbing chains** (churn) don't converge to an interesting distribution —
  instead you compute *expected time to absorption* and *absorption
  probabilities* via the fundamental matrix `N = (I − Q)⁻¹`.
- Monte Carlo simulation is a great way to double-check exact matrix results,
  and generalizes even when the exact math gets messy.
